# RAG Verification Chat (Annual Reports only)
This notebook rebuilds the RAG demo so you can chat with an assistant that only knows the annual reports. If the reports do not contain the answer, it will say so explicitly.


**How this works**
- Loads cached chunks and embeddings from `data/processed`.
- Encodes your question with the same encoder and finds the most similar chunks.
- Answers only when evidence is strong; otherwise responds that it is not in the reports.
- Shows the source file, page, and paragraph for every reply.
- Includes a lightweight chat UI for multi-turn conversations in the notebook.


In [ ]:

# Optional: install runtime dependencies in a fresh environment.
# %pip install sentence-transformers transformers ipywidgets


In [ ]:

from __future__ import annotations

import pickle
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Sequence, Tuple

import numpy as np

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError(
        "Install sentence-transformers (e.g., `pip install sentence-transformers`)."
    ) from exc

PROJECT_ROOT = None
for base in [Path.cwd(), Path.cwd().parent]:
    if (base / "data" / "processed").exists():
        PROJECT_ROOT = base
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find data/processed; run from repo root or notebooks/.")

CACHE_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_PATH = CACHE_DIR / "knowledge_base.pkl"
VECTORS_PATH = CACHE_DIR / "embeddings.npy"

DEFAULT_ENCODER = "all-MiniLM-L6-v2"
TOP_K = 5
MIN_BEST_SCORE = 0.32  # responses are only given when the best chunk clears this

for path in [DOCS_PATH, VECTORS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing cached file: {path}. Run the ingestion notebook first."
        )


In [ ]:

def load_cache() -> Tuple[List[Dict[str, Any]], np.ndarray]:
    with DOCS_PATH.open("rb") as f:
        docs = pickle.load(f)
    vectors = np.load(VECTORS_PATH)
    return docs, vectors


def normalize(mat: np.ndarray) -> np.ndarray:
    return mat / (np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12)


docs, vectors = load_cache()
db_norm = normalize(vectors)
encoder = SentenceTransformer(DEFAULT_ENCODER)

print(f"Loaded {len(docs)} text chunks.")
print(f"Embedding matrix: {vectors.shape}, encoder: {DEFAULT_ENCODER}")


In [ ]:

def retrieve_chunks(question: str, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    query_vec = encoder.encode([question])
    query_vec = query_vec / (np.linalg.norm(query_vec, axis=1, keepdims=True) + 1e-12)
    scores = np.dot(db_norm, query_vec.squeeze())
    order = scores.argsort()[::-1][:top_k]
    results: List[Dict[str, Any]] = []
    for idx in order:
        chunk = docs[idx]
        results.append(
            {
                "text": chunk.get("text", "").strip(),
                "source": chunk.get("source"),
                "page": int(chunk.get("page", -1)),
                "score": float(scores[idx]),
            }
        )
    return results


In [ ]:

USE_LOCAL_MODEL = False  # set to True if you already have the model downloaded locally
LOCAL_MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
local_model = None

if USE_LOCAL_MODEL:
    try:
        from transformers import pipeline

        local_model = pipeline(
            "text-generation",
            model=LOCAL_MODEL_ID,
            tokenizer=LOCAL_MODEL_ID,
            max_new_tokens=180,
            do_sample=False,
            temperature=0.8,
        )
        print(f"Local model ready: {LOCAL_MODEL_ID}")
    except Exception as exc:  # noqa: BLE001
        print(f"Local model not loaded ({exc}). Falling back to evidence-only answers.")
        local_model = None
else:
    print("Running in evidence-only mode (no generative model).")


In [ ]:
        SYSTEM_RULES = (
            "You are a precise assistant that only answers using the provided evidence from annual reports. If the evidence does not support the answer, say I don't know because I couldn't find this in the annual reports.Cite the source file and page when answering."
        )


        def build_prompt(
            history: Sequence[Tuple[str, str]],
            question: str,
            evidence: Sequence[Dict[str, Any]],
            max_chars: int = 400,
        ) -> str:
            hist_lines = [f"{role.upper()}: {msg}" for role, msg in history]
            context_blocks = []
            for i, item in enumerate(evidence, 1):
                txt = item["text"]
                if len(txt) > max_chars:
                    txt = txt[:max_chars] + " ..."
                context_blocks.append(
                    f"[{i}] {item['source']} p.{item['page']} (sim {item['score']:.3f}{txt}"
                )
            context = """.join(context_blocks)history_text = "".join(hist_lines)"""
            
            return (f"{SYSTEM_RULES}"f"{history_text}"f"USER: {question}"f"Evidence:{context}ASSISTANT:")


        def answer_question(
            question: str,
            history: deque | None = None,
            top_k: int = TOP_K,
            min_best_score: float = MIN_BEST_SCORE,
        ) -> Dict[str, Any]:
            history = history or deque(maxlen=8)
            evidence = retrieve_chunks(question, top_k=top_k)
            best = evidence[0]["score"] if evidence else 0.0
            not_found = "I don't know because I couldn't find this in the annual reports."

            if not evidence or best < min_best_score:
                return {"answer": f"{not_found} (best similarity {best:.3f}).", "evidence": evidence}

            if local_model:
                prompt = build_prompt(history, question, evidence)
                try:
                    raw = local_model(prompt)[0]["generated_text"]
                    answer = raw.split("ASSISTANT:")[-1].strip()
                except Exception as exc:  # noqa: BLE001
                    answer = f"{not_found}(model error: {exc})"
            else:
                top = evidence[0]
                snippet = top["text"]
                if len(snippet) > 420:
                    snippet = snippet[:420] + " ..."
                answer = f"{snippet}Source: {top['source']} (page {top['page']})."

            return {"answer": answer, "evidence": evidence}


In [ ]:

        def ask(question: str, history: deque | None = None) -> deque:
            history = history or deque(maxlen=8)
            result = answer_question(question, history=history)
            history.append(("user", question))
            history.append(("assistant", result["answer"]))

            print(f"Q: {question}A: {result['answer']}")

            for i, ev in enumerate(result["evidence"], 1):
                snippet = ev["text"]
                if len(snippet) > 200:
                    snippet = snippet[:200] + " ..."
                print(f"[{i}] {ev['source']} p.{ev['page']} (sim={ev['score']:.3f}){snippet}")
            return history


Try the helper below for quick, one-off questions. Questions that are not in the reports should return a clear "I don't know" response.


In [ ]:

# Example usage (uncomment to run):
# chat_history = ask("What are Amundi's assets under management?")
# chat_history = ask("What did Picasso paint?", history=chat_history)


## Interactive chat
Run the next cell to start a multi-turn chat UI. The assistant always shows the evidence it used (file and page). Stop the chat by clearing the output or restarting the kernel.


In [ ]:
import ipywidgets as widgets
from IPython.display import Markdown, display


def start_chat(top_k: int = TOP_K):
    history = deque(maxlen=8)
    output = widgets.Output()
    question_box = widgets.Text(placeholder="Ask about the annual reports...")
    # Ensure text changes only fire on submit/focus-out (not per keystroke)
    question_box.continuous_update = False
    send_btn = widgets.Button(description="Send", button_style="primary")
    status = widgets.HTML(
        value=(
            f"Mode: {'LLM-guided' if local_model else 'evidence-only'} · "
            f"Top-k={top_k} · Min score={MIN_BEST_SCORE}"
        )
    )

    def process_question():
        question = question_box.value.strip()
        if not question:
            return
        question_box.value = ""

        result = answer_question(
            question,
            history=history,
            top_k=top_k,
            min_best_score=MIN_BEST_SCORE,
        )
        history.append(("user", question))
        history.append(("assistant", result["answer"]))

        with output:
            display(Markdown(f"**You:** {question}"))
            display(Markdown(f"**Bot:** {result['answer']}"))
            for i, ev in enumerate(result["evidence"], 1):
                snippet = ev["text"]
                if len(snippet) > 320:
                    snippet = snippet[:320] + " ..."
                display(
                    Markdown(
                        f"> [Evidence {i} | {ev['source']} p.{ev['page']} | sim={ev['score']:.3f}] {snippet}"
                    )
                )
            display(Markdown("---"))

    def handle_click(_):
        process_question()

    def handle_submit(change):
        # Only process real value changes (enter key or focus-out).
        if change.get("name") == "value" and change.get("type") == "change":
            process_question()

    send_btn.on_click(handle_click)
    question_box.observe(handle_submit, names="value")
    ui = widgets.VBox([status, widgets.HBox([question_box, send_btn]), output])
    display(ui)


start_chat()


In [ ]:
start_chat()